In [ ]:
# v1
# v2 : added "if it likes it then has a higher chnace of liking it" (i assume randomness too much)
# v3 : changed it to +-0.1 pet like dislike (slow but nice randomness was too much i think?)
# v4 : just a combined plot +-1 yes
# v5 : sigmoid for liking chance normalization for friendliness, +0.3 -0.5 , and decay of 0.5% per tick 

In [28]:
import csv
import matplotlib.pyplot as plt
import pandas as pd

##
df = pd.read_csv('activity_finished.csv', header=None, 
                 names=['timestamp' , 'pet_name', 'activity_name', 'partner_name', 'liked', 'activity_relationship', 'partner_relationship'])

df_activity_finished = df
# df_activity_finished['timestamp'] = pd.to_datetime(df_activity_finished['timestamp'], unit="ms")
df_activity_finished["partner_name"] = (
    df_activity_finished["partner_name"]
    .str.split("@", n=1)
    .str[-1] # I love you python
)

##
df = pd.read_csv('relationships.csv', header=None,
                 names=['timestamp' , 'pet_name', 'other_entity_id', 'friendliness'])
df_relationships = df
df_relationships['timestamp'] = pd.to_datetime(df_relationships['timestamp'], unit="ms")
df_relationships["other_entity_id"] = (
    df_relationships["other_entity_id"]
    .str.split("@", n=1)
    .str[-1]
)

In [26]:
sum(s in "alice -> mom" for s in ["alice", "brice", "ant"]) # i love you python pt 2

1

In [37]:
df = df_activity_finished.copy()
df = df.dropna(subset=["pet_name", "partner_name"])
df["pair_key"] = df.apply(
    lambda r: tuple(sorted((r.pet_name, r.partner_name))),
    axis=1
)

df = df.sort_values(["pair_key", "activity_name", "timestamp"])
# df["cluster"] = (
#     df.groupby(["pair_key", "activity_name"])["timestamp"] 
#     .diff().lt(100).fillna(True).cumsum()
# )

# df["cluster"]
df["cluster"] = (df.groupby(["pair_key", "activity_name"])["timestamp"].diff().gt(100).fillna(True).cumsum())
df

,timestamp,pet_name,activity_name,partner_name,liked,activity_relationship,partner_relationship,pair_key,cluster
83,1785131804610,Bat,Dance,Ant,True,0.145534,0.500000,"(Ant, Bat)",0
52,1785131796487,Ant,Dance,Bee,False,-0.813651,-0.922265,"(Ant, Bee)",0
53,1785131796487,Bee,Dance,Ant,False,-0.271812,-0.935451,"(Ant, Bee)",0
82,1785131804610,Ant,Dance,Bee,True,-0.111440,-0.193061,"(Ant, Bee)",1
21,1785131785637,Ant,Fight,Bee,False,-0.600000,-0.600000,"(Ant, Bee)",1
...,...,...,...,...,...,...,...,...,...
61,1785131797943,brice,Dance,alice,True,0.695833,0.500000,"(alice, brice)",12
74,1785131801499,brice,Dance,alice,True,1.089748,0.923771,"(alice, brice)",13
89,1785131805651,brice,Dance,alice,False,0.274061,0.140935,"(alice, brice)",14
72,1785131801499,bear,Dance,brice,False,-0.248551,-0.600000,"(bear, brice)",14


In [ ]:
pet_names = df_relationships["pet_name"].unique()
df_relationships['pair'] = df_relationships.apply(lambda row: f"{row['pet_name']} → {row['other_entity_id']}", axis=1)
df_relationships_pivot = df_relationships.pivot(
    index='pair',
    columns='timestamp',
    values='friendliness'
)

df_relationships_pivot = df_relationships_pivot.fillna(0)

In [ ]:
mask = df_relationships_pivot.index.map(
    lambda a: (sum(s in a for s in pet_names) >= 2)
)

heat = df_relationships_pivot[mask]
heat = heat.sort_index(axis=1)

import plotly.express as px

fig = px.imshow(
    heat,
    aspect="auto",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    labels={
        "x": "Time",
        "y": "Relationship",
        "color": "Value"
    }
)

group_size = 5 

for i in range(group_size, len(heat.index), group_size):
    fig.add_hline(
        y=i - 0.5,
        line_width=5,
        line_color="black"
    )

# fig.add_scatter(
#     x=df['timestamp'],
#     y=df['pair'],
#     mode='markers',
#     marker=dict(
#         symbol='diamond',
#         size=3,
#         color='black'
#     ),
#     hovertext=df['activity_name'],
#     hoverinfo='text'
# )
# fig.update_xaxes(range=[heat.columns.min(), heat.columns.max()])
# fig.update_yaxes(range= [heat.index.min(), heat.index.max()])

# fig.show()

fig.update_layout(height=900)

fig.show()

In [ ]:
df["activity_pair"] = df["pet_name"] + " : " + df["activity_name"]

heat = df.pivot_table(
    index="activity_pair",
    columns="timestamp",
    values="activity_relationship",
    aggfunc="last"
)

heat = heat.sort_index(axis=1).ffill(axis=1)
heat.fillna(0, inplace=True)

fig = px.imshow(
    heat,
    aspect="auto",
    color_continuous_scale="RdYlGn",
)

fig.update_layout(height=900)
fig.show()